In [1]:

from pprint import pprint

from ollama import chat
from tools import Top_Type, code_extraction, equi_check, linter, netlist
# from typing import TypedDict
# from langgraph.graph import START,StateGraph,END


In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

# Checking for the presence of model and loading  the model 


In [3]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog and symbolic model verification coding ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

As Qwen, I was last updated in October 2023, so my knowledge cut-off is up until that point. This means I can provide information and assistance based on what was known up to that time.

Regarding Verilog and symbolic model verification coding, I am well-versed with the syntax and features of Verilog, which is a hardware description language commonly used for designing digital circuits. If you have specific questions or need help with Verilog code, I can certainly assist you!

Symbolic model verification is an advanced technique that allows for the analysis of large design spaces without simulating every possible state. It's particularly useful in verifying complex systems where exhaustive simulation would be impractical. If you have questions about how to approach symbolic model verification or need help with specific coding aspects, feel free to ask!


In [4]:
# SPECIFICATION_SYSTEM_PROMPT = r"""
# R — ROLE
# --------
# You are a Hardware Design Specification Provider.

# Convert the user's natural-language RTL request into a structured,
# detailed specification for a separate Verilog-2001 code-generation LLM.

# You do NOT generate RTL.

# I — INSTRUCTIONS
# -----------------
# Extract the intended:

# - design/function
# - combinational or sequential behavior
# - inputs/outputs, widths and signedness
# - clock/reset requirements
# - protocol/interface behavior
# - operations and data flow
# - state/FSM behavior
# - timing/latency requirements
# - important corner cases

# Do not invent critical functionality. Clearly identify assumptions
# and missing information.

# S — STEPS
# ---------
# 1. Identify the design and its purpose.
# 2. Extract the complete interface.
# 3. Determine combinational/sequential behavior.
# 4. Define functional and control behavior.
# 5. Define clock/reset behavior.
# 6. Define states/protocol behavior where applicable.
# 7. Identify assumptions, ambiguities and corner cases.
# 8. Provide implementation guidance for the RTL generator.

# E — EXPECTED OUTPUT
# -------------------
# Return ONLY valid JSON:

# {
#   "design": {
#     "name": "",
#     "type": "COMBINATIONAL | SEQUENTIAL",
#     "purpose": "",
#     "description": ""
#   },

#   "interface": [
#     {
#       "name": "",
#       "direction": "input | output | inout",
#       "width": "",
#       "signed": "yes | no | unspecified",
#       "description": ""
#     }
#   ],

#   "clock_reset": {
#     "clock": "",
#     "frequency": "",
#     "period": "",
#     "reset": "",
#     "reset_type": "synchronous | asynchronous | unspecified",
#     "polarity": "active_high | active_low | unspecified"
#   },

#   "functional_specification": {
#     "main_function": "",
#     "operations": [],
#     "data_flow": [],
#     "control_flow": [],
#     "state_behavior": [],
#     "protocol_behavior": [],
#     "latency": ""
#   },

#   "corner_cases": [],
#   "assumptions": [],
#   "clarifications": [],

#   "code_generation_guidance": {
#     "language": "Verilog-2001",
#     "module_name": "",
#     "requirements": []
#   }
# }

# N — NARROW CONSTRAINTS
# ----------------------
# 1. Output ONLY JSON.
# 2. Do not generate RTL or testbench code.
# 3. Do not discuss tools or verification flows.
# 4. Do not invent unspecified critical behavior.
# 5. Separate explicit requirements from assumptions.
# 6. Preserve all important information from the user.
# 7. Make the functional specification detailed enough for the RTL
#    generator to implement it without reinterpreting the original prompt.
# 8. Use Verilog-2001; do not specify SystemVerilog constructs.
# 9. Do not output hidden reasoning or chain-of-thought.
# 10. Keep the output focused on the design specification.
# """

In [5]:
SPECIFICATION_SYSTEM_PROMPT = r"""
ROLE  
You are an RTL specification extractor.

INSTRUCTIONS  
Convert the user's request into the minimum information required to generate synthesizable Verilog-2001 RTL.  
Extract only explicitly stated requirements. Never invent behavior.

STEPS  
1. Identify module name and COMBINATIONAL/SEQUENTIAL type.  
2. Extract exact ports, directions, widths and signedness when stated.  
3. Extract clock/reset only when stated.  
4. Convert required functionality into short, unambiguous behavior rules.  
5. List FSM states only when explicitly required or clearly defined.  
6. Preserve numeric parameters exactly.

END GOAL  
Return one compact machine-readable RTL contract for the code generator.

NARROWING CONSTRAINTS  
- JSON only; no Markdown or explanation.
- Do not add assumptions, corner cases, optimizations, timing, latency, protocols or error handling unless explicitly specified.
- Do not duplicate information.
- Do not infer missing widths, resets, states or functionality.
- Unknown values = null.
- Use exact user signal/module names.
- Each behavior item must describe one implementation requirement.
- No RTL/code generation.

OUTPUT
{
  "module": "",
  "type": "COMBINATIONAL|SEQUENTIAL",
  "ports": [
    {"name":"","dir":"input|output|inout","width":null,"signed":null}
  ],
  "clock": {"name":null,"edge":null},
  "reset": {"name":null,"edge":null,"async":null},
  "parameters": {},
  "behavior": [],
  "states": []
}
"""

In [6]:
def spec_analyzer(prompt:str):
    stream=chat(model=Ollama_MODEL,messages=[{"role":"System",
                                              "content":f""" {SPECIFICATION_SYSTEM_PROMPT} """},
                                              {"role":"user",
                                               "content":f"{prompt}"}],
                                               stream=True,
                                               options={
                                                        "temperature":0.25,
                                                        "top_p":0.9,
                                                        "top_k":20,
                                                        "seed":0
                                                      })
    respon=""
    for chunk in stream:
        content=chunk["message"]["content"] 
        print(content,end="",flush=True)
        respon+=content
    return respon

In [7]:
_VERILOG_SYSTEM =""" 
ROLE
You are a senior RTL design engineer specializing in synthesizable Verilog-2001.

INSTRUCTIONS
Implement the user's specification exactly. Generate clean, synthesizable RTL.
Use Verilog-2001 only and return only the complete Verilog code.

STEPS

1. Identify all signals, their widths, signedness, clocks, resets, and required behavior.
2. Generate the RTL using correct sequential and combinational coding styles.
3. Before returning, check syntax, widths, constants, signedness, bit selections, and synthesis safety.

END GOAL
Produce Verilator/Icarus-compatible Verilog-2001 RTL with no width or syntax errors.

NARROW CONSTRAINTS

* Use explicit signal widths.
* Every sized numeric literal must have enough bits for all specified digits.
  Example: 3'b111, NOT 2'b111.
* Match constant widths to the destination/comparison signal where practical.
  Example: reg [2:0] count → 3'b111 or 3'd7.
* Never select bits outside a signal's declared range.
* Use '=' in always @(*) and '<=' in clocked always blocks.
* Give combinational outputs a value on every path.
* Include default in case statements.
* Use $signed/$unsigned when signedness matters.
* No SystemVerilog: logic, always_comb, always_ff, typedef, enum, struct, interface, class, etc.
* Do not invent requirements or modify the user's specification.
* Return ONLY Verilog-2001 code; no Markdown or explanation.

"""

In [8]:
def code_gen(prompt:str)->str:
    """ 
    INPUT:
    prompt: excepts user promtp a string providing the detials about the verilog code to be written by the llm.

    OUTPTS:
        gives the code as output 
    """
    stream=chat(model=Ollama_MODEL,messages=[
                                                {
                                                    "role": "system",
                                                    "content": _VERILOG_SYSTEM
                                                },
                                                {
                                                    "role": "user",
                                                    "content": prompt
                                                }
                                            ],
                                            stream=True,
                                            options={
                                                    "temperature":0.45,
                                                     "top_p":0.8,
                                                     "top_k":20,
                                                     "seed":0
                                                     })
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [9]:
User_inpt=input("Prompt the verilog module  requesting")
spec_resp=spec_analyzer(User_inpt)


```json
{
  "module": "ALU",
  "type": "COMBINATIONAL",
  "ports": [
    {"name":"A","dir":"input","width":32,"signed":true},
    {"name":"B","dir":"input","width":32,"signed":true},
    {"name":"op","dir":"input","width":4,"signed":false},
    {"name":"result","dir":"output","width":32,"signed":true}
  ],
  "clock": null,
  "reset": null,
  "parameters": {},
  "behavior": [
    {"case":"op == 4'b0000","action":"result = A + B"},
    {"case":"op == 4'b0001","action":"result = A - B"},
    {"case":"op == 4'b0010","action":"result = A * B"},
    {"case":"op == 4'b0011","action":"result = A / B"}
  ],
  "states": []
}
```

In [10]:
response=code_gen(spec_resp)

```verilog
module ALU (
  input [31:0] A,
  input [31:0] B,
  input [3:0] op,
  output reg [31:0] result
);

always @(*) begin
  case (op)
    4'b0000: result = A + B;
    4'b0001: result = A - B;
    4'b0010: result = A * B;
    4'b0011: if (B != 32'd0) result = A / B; else result = 32'd0; // Handle division by zero
    default: result = 32'd0; // Default case to avoid undefined behavior
  endcase
end

endmodule
```

In [11]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module ALU (
  input [31:0] A,
  input [31:0] B,
  input [3:0] op,
  output reg [31:0] result
);

always @(*) begin
  case (op)
    4'b0000: result = A + B;
    4'b0001: result = A - B;
    4'b0010: result = A * B;
    4'b0011: if (B != 32'd0) result = A / B; else result = 32'd0; // Handle division by zero
    default: result = 32'd0; // Default case to avoid undefined behavior
  endcase
end

endmodule


In [12]:
res=linter(clean_Vcode)
print(res)

{'return_code': 0, 'stdout': '- V e r i l a t i o n   R e p o r t: Verilator 5.032 2025-01-01 rev (Debian 5.032-1)\n- Verilator: Built from 0.020 MB sources in 2 modules, into 0.017 MB in 5 C++ files needing 0.000 MB\n- Verilator: Walltime 0.033 s (elab=0.006, cvt=0.010, bld=0.000); cpu 0.007 s on 1 threads; alloced 7.863 MB\n', 'stderr': ''}


In [13]:
top=Top_Type(clean_Vcode)

In [14]:
pprint(top)

{'err': '',
 'output': '\n'
           ' '
           '/----------------------------------------------------------------------------\\\n'
           ' |  yosys -- Yosys Open SYnthesis '
           'Suite                                       |\n'
           ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
           '<claire@yosyshq.com>         |\n'
           ' |  Distributed under an ISC-like license, type "license" to see '
           'terms        |\n'
           ' '
           '\\----------------------------------------------------------------------------/\n'
           ' Yosys 0.62+112 (git sha1 687a36af3, g++ 13.3.0-6ubuntu2~24.04.1 '
           '-fPIC -O3)\n'
           '\n'
           '-- Running command ` \n'
           '                             read_verilog /tmp/tmp5mird8bk.v;\n'
           '                             proc;\n'
           '                             stat;\n'
           "                            ' --\n"
           '\n'
           '1. Executing V

In [15]:
top_name=top['top_module']
print(top_name)

ALU


In [16]:
net_out=netlist(top=top_name,module_path=f"./src/{top_name}.v",lib_path="../Nangate45/Nangate45_typ.lib")

In [17]:
pprint(net_out)

{'netlist_returncode': 0,
 'netlist_stderr': '',
 'netlist_stdout': '\n'
                   ' '
                   '/----------------------------------------------------------------------------\\\n'
                   ' |  yosys -- Yosys Open SYnthesis '
                   'Suite                                       |\n'
                   ' |  Copyright (C) 2012 - 2026  Claire Xenia Wolf '
                   '<claire@yosyshq.com>         |\n'
                   ' |  Distributed under an ISC-like license, type "license" '
                   'to see terms        |\n'
                   ' '
                   '\\----------------------------------------------------------------------------/\n'
                   ' Yosys 0.62+112 (git sha1 687a36af3, g++ '
                   '13.3.0-6ubuntu2~24.04.1 -fPIC -O3)\n'
                   '\n'
                   '-- Running command `\n'
                   '            read_verilog ./src/ALU.v;\n'
                   '            hierarchy -top ALU

In [ ]:
path_1=f"./src/{top_name}.v"
path_2=f"./src/{top_name}_netlist.v"
des_type=top["type"]
equi_out=equi_check(top=top_name,path_1=path_1,path_2=path_2,design_type=des_type)

In [1]:
pprint(equi_out)

Pretty printing has been turned OFF
